# 03 · 剪枝对比（E2 / E3 / E5）

所有方法从**同一个 Dense-Aug checkpoint** 出发，逐层非结构化剪枝（每层相同稀疏度），用同一批打分原图、同一批评估对和固定遮挡。

| 方法 | 打分数据 | 反传次数 |
|---|---|---|
| Random | 随机分数（按打分种子） | 0 |
| Magnitude | \|w\| | 0 |
| Clean-Taylor | B 张干净图 | B |
| Occ-Taylor | B 张遮挡图 | B |
| Mixed-Taylor | B/2 干净 + B/2 遮挡，直接平均 | B |
| OAGP | 与 Mixed 相同的 B 个视图；每层按均值归一化后加权平均（α） | B |
| OAGP-max | 同上，逐权重取较大者 | B |

**共享校准**：B=1024 张训练原图分成 A、B 两半，一次性算 4 组逐样本 Taylor 分数（A 干净、B 干净、A 遮挡、B 遮挡，共 2048 次反传，约 30 秒）。Clean = A干净+B干净，Occ = A遮挡+B遮挡，Mixed / OAGP / OAGP-max 都用 A干净 + B遮挡 —— 三者看到的视图完全相同，只差合并方式。

**流程**：先只在 CASIA 留出验证对上跑（`TEST_SETS = ()`）选稀疏度和 α；设置冻结后再加 `("lfw",)` 得到最终测试结果。微调用独立的小学习率（默认 0.01，cosine 到 0），每步后重新施加掩码。

In [ ]:
from pathlib import Path
import os, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/occprune").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
from occprune.prune_run import run_pruning, METHODS
from occprune.prune_report import load_runs, metrics_table, summary_table, paired_bootstrap
print("methods:", METHODS)

## 1. 设置
- `TAG` 区分一组实验（输出在 `results/pruning/runs/<TAG>_score<seed>/`，已存在的目录不会被覆盖，会直接读取）。
- E2 稀疏度扫描：`METHODS = ["magnitude"]`，`SPARSITIES = [0.5, 0.6, 0.7, 0.8, 0.9]`，`FT_EPOCHS = 0`。
- E3 主对比：全部方法、拐点附近 2 个稀疏度，`FT_EPOCHS = 0` 和 `1` 各跑一组，打分种子 42/43/44。
- 正式结果统一在一台机器上跑（不同 GPU / MPS 之间会有约 0.5 个百分点的差异）。

In [ ]:
CHECKPOINT = "checkpoints/dense_aug_seed42.pt"
TAG = "e2_magnitude_sweep"
METHODS_RUN = ["magnitude"]            # 或 list(METHODS)
SPARSITIES = [0.5, 0.6, 0.7, 0.8, 0.9]
SCORING_SEEDS = [42]                   # E3: [42, 43, 44]
FT_EPOCHS = 0                          # E3 微调: 1
TEST_SETS = ()                         # 设置冻结后改为 ("lfw",)
RUN = False                            # True = 运行缺失的实验

run_dirs = []
for seed in SCORING_SEEDS:
    dest = Path("results/pruning/runs") / f"{TAG}_score{seed}"
    run_dirs.append(str(dest))
    if (dest / "result.json").is_file():
        print("exists:", dest)
    elif RUN:
        run_pruning(CHECKPOINT, dest, methods=METHODS_RUN, sparsities=SPARSITIES, scoring_seed=seed,
                    ft_epochs=FT_EPOCHS, test_sets=TEST_SETS)
    else:
        print("missing (set RUN = True):", dest)

## 2. 结果表
准确率（%）按打分种子取平均；`region_mean` 是双眼、口鼻、上半脸、下半脸四种区域遮挡的平均。`dense` 行是未剪枝的参照。

In [ ]:
existing = [d for d in run_dirs if (Path(d) / "result.json").is_file()]
DATASET = "lfw" if TEST_SETS else "validation"
if existing:
    df = metrics_table(load_runs(existing))
    display(summary_table(df, dataset=DATASET, stage="one_shot").round(2))
else:
    df = None
    print("no finished runs yet")

In [ ]:
# 微调后的结果（FT_EPOCHS > 0 时）
if df is not None and (df.stage == "fine_tuned").any():
    display(summary_table(df, dataset=DATASET, stage="fine_tuned").round(2))

## 3. 两个方法的配对比较
同一批评估对上，各自用 10 折阈值判断每一对，再对**对**重采样，得到准确率差（B − A）的 95% 置信区间。区间跨 0 时不下结论。

In [ ]:
if existing and {"oagp", "oagp_max"} <= set(METHODS_RUN):
    display(paired_bootstrap(existing[0], "oagp", "oagp_max", sparsity=SPARSITIES[0], dataset=DATASET).round(2))